In [20]:
import requests

url = "https://raw.githubusercontent.com/apache/ossie/main/examples/tpcds_semantic_model.yaml"

response = requests.get(url)
osi = response.text
#print(osi)

In [27]:
%reload_ext autoreload
%autoreload 0

import importlib
import os
from pathlib import Path

# TOM assemblies live at the package root, but this notebook's cwd is tests/.
os.environ["OSSIE_MICROSOFT_TOM_ASSEMBLIES"] = str(
    Path.cwd().parent / ".tom" / "assemblies"
)

_sql_to_dax = importlib.reload(
    importlib.import_module("ossie_microsoft._sql_to_dax")
)
_tom = importlib.reload(importlib.import_module("ossie_microsoft.tom"))
_converter = importlib.reload(
    importlib.import_module("ossie_microsoft.ossie_to_semantic_model")
)
convert_ossie_to_semantic_model = _converter.convert_ossie_to_semantic_model

print(f"Loaded converter from: {_converter.__file__}")


Loaded converter from: C:\Users\mikova\ApacheOssie\ossie\converters\microsoft\src\ossie_microsoft\ossie_to_semantic_model.py


In [ ]:
workspace_id = "49bd15b9-0a7e-4e41-94ae-ee5d8e8d1990" # ID of workspace
item_id = "6703e922-40a2-4fb1-86a2-f4c02784dd96" # ID of Lakehouse
x = convert_ossie_to_semantic_model(
    ossie_yaml_str=osi,
    source={
        "workspaceId": workspace_id,
        "itemId": item_id,
    },
    output_format="TMDL" # Can be "TMSL" or "TMDL"
)

[model] custom_extensions for vendor 'SALESFORCE' have no Power BI equivalent and are dropped
[model] custom_extensions for vendor 'DBT' have no Power BI equivalent and are dropped
[dataset 'store_sales'] Power BI has no composite key; primary key (ss_item_sk, ss_ticket_number) is not marked on the table
[dataset 'store_sales'] Power BI has no composite unique constraint; unique key (ss_item_sk, ss_ticket_number) is not marked on the table
[metric 'total_sales'] no home table recorded; the measure is placed on 'store_sales'
[metric 'total_sales'] Power BI infers a measure's data type from its DAX expression, so datatype 'Decimal' is not applied
[metric 'total_profit'] no home table recorded; the measure is placed on 'store_sales'
[metric 'total_profit'] Power BI infers a measure's data type from its DAX expression, so datatype 'Decimal' is not applied
[metric 'customer_lifetime_value'] no home table recorded; the measure is placed on 'store_sales'
[metric 'customer_lifetime_value'] Pow

In [35]:
print(x)

database tpcds_retail_model
	compatibilityLevel: 1702

	/// TPC-DS retail semantic model for sales and customer analytics
	model Model
		culture: en-US

		/// Fact table containing all store sales transactions
		table store_sales

			/// Total sales revenue across all transactions
			measure total_sales = SUM('store_sales'[ss_ext_sales_price])

				annotation OssieExpressionDialect = ANSI_SQL

				annotation OssieExpression = SUM(store_sales.ss_ext_sales_price)

				annotation OssieAIContext = {"synonyms": ["total revenue", "gross sales", "sales amount"]}

			/// Total net profit from store sales
			measure total_profit = SUM('store_sales'[ss_net_profit])

				annotation OssieExpressionDialect = ANSI_SQL

				annotation OssieExpression = SUM(store_sales.ss_net_profit)

				annotation OssieAIContext = {"synonyms": ["net profit", "total earnings", "profit"]}

			/// Average lifetime sales value per customer
			measure customer_lifetime_value = DIVIDE(SUM('store_sales'[ss_ext_sales_price]